# Phase 3 — self-distillation round 1 (one launch: harvest → train → evaluate)

Fresh Colab, **A100 + High-RAM**, then **Run All**. This runs a full iterated-distillation round end
to end and reports whether it beat the P6 baselines. Set the toggles in CELL 3, then walk away.

**Toggles (CELL 3):** `SMOKE=True` first (fast end-to-end plumbing check, ~20-30 min). Once it's clean,
set `SMOKE=False` and re-Run-All for the real **overnight** round (harvest ~2641 rows × N + a full
2-epoch retrain → several hours). `TAU=0.30` is set from the oracle@N result — lower it if CELL 3's
smoke shows very few rows `replaced`.

**Decision gate (CELL 5):** compare to the P6 baselines —
- free-running **greedy behavioral fidelity > 0.159**, AND
- **oracle@32 on distill_r1 > 0.30** (the P6 coverage ceiling)
→ the distribution moved → **iterate** (re-run with `--adapter models/sft_adapters/distill_r1_smokefull`).
If oracle stays ~0.30 → distillation plateaued → escalate to RL/GRPO.


In [7]:
# CELL 1 - provision (clone, install, HF token, stage the corpus)
import os, pathlib, subprocess, sys
REPO, BRANCH = '/content/SLM', 'feat/two-stage'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
# ALWAYS sync to the latest branch HEAD (cheap) so an existing runtime / re-run can never execute
# stale code. The previous version guarded the git pull behind SLM_PROVISIONED, so a runtime first
# provisioned on an older commit kept running it. Only the slow pip install stays guarded.
subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
if not os.environ.get('SLM_PROVISIONED'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                if _l.strip().startswith(name + '='):
                    return _l.split('=', 1)[1].strip().strip('"').strip("'")
    return None
import getpass
for _n, _p in (('HF_TOKEN', 'HF_TOKEN (read): '), ('HF_WRITE_TOKEN', 'HF_WRITE_TOKEN (SLM_Alpha_Write, optional): ')):
    if not os.environ.get(_n):
        os.environ[_n] = _env_token(_n) or (getpass.getpass(_p).strip() if _n == 'HF_TOKEN' else (_env_token(_n) or ''))
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('staging ~9.85GB corpus ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged')


HEAD: 91b1fc7 Add phase3_distill.ipynb — one-launch distillation round (harvest→train→eval)
HF_TOKEN set: True
corpus already staged


In [8]:
# CELL 2 - build base_resized + download the P6 two-stage adapter (the harvest model)
import os, pathlib, subprocess, sys, json
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
D = pathlib.Path('models/base_resized')
if not ((D / 'vocab_resize_manifest.json').is_file() and (D / 'preprocessor_config.json').is_file()):
    subprocess.run([sys.executable, '-m', 'sft.vocab_resize', '--out', 'models/base_resized'],
                   check=True, env={**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'})
vr = json.load(open(D / 'vocab_resize_manifest.json'))
assert vr.get('tokenizer_version') == 'vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f'
from huggingface_hub import snapshot_download
P6 = 'p6_twostage_d0f9c744_smokefull'
snapshot_download(repo_id='ericrcwu/LUT_SLM_sft_adapters', allow_patterns=[P6 + '/*'],
                  local_dir='models/sft_adapters', token=os.environ.get('HF_TOKEN'))
assert pathlib.Path('models/sft_adapters', P6, 'adapter_model.safetensors').is_file()
print('base_resized + P6 adapter ready')


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

base_resized + P6 adapter ready


In [ ]:
# CELL 3 - harvest the distilled corpus (best-of-N winners >= TAU rewrite the training targets)
import os, sys, subprocess
SMOKE = True          # True: fast plumbing check (~200 rows). False: full overnight harvest.
TAU = 0.30            # absolute fidelity bar; lower if too few rows are 'replaced'
N = 16                # best-of-N samples per training row
HARVEST = 'models/sft_adapters/p6_twostage_d0f9c744_smokefull'   # round 1: P6; round 2: distill_r1_*

env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
cmd = [sys.executable, '-m', 'scripts.build_distillation_corpus',
       '--adapter', HARVEST, '--tau', str(TAU), '--n', str(N),
       '--source-rows', 'data/active_sft/active_rows.jsonl',
       '--out-dir', 'data/active_sft_distilled']
if SMOKE:
    cmd += ['--limit', '200']
print('running:', ' '.join(cmd), '\n')
r = subprocess.run(cmd, env=env)
assert r.returncode == 0, 'harvest failed'
# the printed line "[distill] DONE counts=... mean_best_of_N=..." shows replaced/kept_gold — sanity-check
# that a meaningful fraction was replaced (else lower TAU and re-run this cell).


running: /usr/bin/python3 -m scripts.build_distillation_corpus --adapter models/sft_adapters/p6_twostage_d0f9c744_smokefull --tau 0.3 --n 16 --source-rows data/active_sft/active_rows.jsonl --out-dir data/active_sft_distilled --limit 200 



In [ ]:
# CELL 4 - retrain a fresh adapter on the distilled corpus (locked knobs; 2 epochs)
import os, sys, subprocess
SMOKE = True             # keep in sync with CELL 3
GRAD_CHECKPOINT = True   # False = ~1.5-2x faster but much more VRAM (may OOM on a 40GB A100;
                         # gradients are identical either way, so results are unchanged)
env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
cmd = [sys.executable, '-m', 'sft.train',
       '--config', 'configs/candidate_distill.json',
       '--resized-model', 'models/base_resized',
       '--smoke-size', '200' if SMOKE else '0',
       '--run-id', 'distill_r1']
if not GRAD_CHECKPOINT:
    cmd.append('--no-gradient-checkpointing')
print('running:', ' '.join(cmd), '\n')
r = subprocess.run(cmd, env=env)
assert r.returncode == 0, 'train failed'
import glob
ADAPTER = sorted(glob.glob('models/sft_adapters/distill_r1_smoke*'))[-1]
print('trained adapter:', ADAPTER)


In [ ]:
# CELL 5 - evaluate distill_r1 on the UNTOUCHED holdout vs the P6 baselines
import os, sys, subprocess, glob, json
SMOKE = True
env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
ADAPTER = sorted(glob.glob('models/sft_adapters/distill_r1_smoke*'))[-1]
blim = '16' if SMOKE else '64'
olim, on = ('16', '16') if SMOKE else ('32', '32')

print('=== behavioral fidelity (greedy vs sampling) ===')
subprocess.run([sys.executable, '-m', 'sft.score_tokens', '--config', 'configs/candidate_distill.json',
                '--adapter', ADAPTER, '--behavioral-sampling', 'both', '--behavioral-limit', blim], env=env)
print('\n=== oracle@N on distill_r1 (did coverage climb past P6 0.30?) ===')
subprocess.run([sys.executable, '-m', 'eval.oracle_at_n', '--config', 'configs/candidate_distill.json',
                '--adapter', ADAPTER, '--limit', olim, '--n', on], env=env)
print('\nGATE: greedy behavioral fidelity > 0.159 AND oracle@32 > 0.30 => distribution moved => ITERATE')
print('      (re-run CELL 3 with --adapter', ADAPTER, '). If oracle ~0.30 => plateau => escalate to RL.')
